# **🤖 RAG Agent for Research Papers**

## 1. 🔑 API Key Configuration

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import huggingface

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")
hf_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")

## 2. ⚙️ LLM & Embedding Model Setup

In [2]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(
    model="gpt-4o-mini",
    api_key= openai_key)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-base-en-v1.5")

x:\Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
x:\Project\venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 845.14it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.

## 3. 📦 Import Libraries

In [ ]:
from llama_index.core.node_parser import SemanticSplitterNodeParser,SentenceSplitter
from llama_index.core import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex , SummaryIndex
import nest_asyncio
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from typing import List
from llama_index.core.vector_stores import FilterCondition
from llama_cloud import MetadataFilters
from llama_index.core.vector_stores import MetadataFilter ,ExactMatchFilter
from llama_index.core.tools import FunctionTool
from llama_index.llms.openai import OpenAI
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.memory import ChatMemoryBuffer

nest_asyncio.apply()


## 4. 📄 Document Loading & Chunking

In [ ]:
document=SimpleDirectoryReader(input_files=["File path"]).load_data()
splitter=SemanticSplitterNodeParser(embed_model=Settings.embed_model,breakpoint_percentile_threshold=75)
node=splitter.get_nodes_from_documents(document)

filtered_node = [
    n for n in node
    if len(n.get_content()) > 150
]

secondary_splitter=SentenceSplitter(
    chunk_size=600,
    chunk_overlap=100
)
final_node=[]

for n in filtered_node:
    if len(n.get_content()) > 900:
        new_node = secondary_splitter.get_nodes_from_documents([n])
        final_node.extend(new_node)
    else:
        final_node.append(n)


vector_store=VectorStoreIndex(final_node)
summary_store = SummaryIndex(final_node)

## 5. 🛠️ Query Engines & Tool Definitions

In [5]:
summary_query_engine=summary_store.as_query_engine(
    response_mode="tree_summarize",
    use_async=True
)

vector_query_engine=vector_store.as_query_engine()


def vector_query(
        query:str,
        page_number:List[str]
)->str:
    """"Perform a vector search over an index.
    
    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.
    """
    if page_number:
        query_engine = vector_store.as_query_engine(
            similarity_top_k=2,
            filters=MetadataFilters(          
                filters=[
                    ExactMatchFilter(key="page_label", value=p)
                    for p in page_number     
                ],
                condition=FilterCondition.OR,
            ),
        )
    
    else:
        query_engine = vector_store.as_query_engine(similarity_top_k=2)
    
    response = query_engine.query(query)  
    return response

    


vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)


summary_tool = QueryEngineTool(
        query_engine=summary_query_engine,
        metadata=ToolMetadata(
            name="paper_summarizer",
            description=(
                "Use this to summarize an entire paper or section. "
                "Good for: 'summarize this paper', 'what is the abstract about', "
                "'give me an overview of the findings'."
            ),
        ),
    )


#Optional 
"""specific_tool=QueryEngineTool(
    query_engine=vector_query_engine,
    metadata=ToolMetadata(
        name="specific search",
        description=(
            "Useful for retrieving specific context from the research paper. "
            "Use for detailed questions about methods, results, experiments, or specific sections."
            )
        )
)"""





'specific_tool=QueryEngineTool(\n    query_engine=vector_query_engine,\n    metadata=ToolMetadata(\n        name="specific search",\n        description=(\n            "Useful for retrieving specific context from the research paper. "\n            "Use for detailed questions about methods, results, experiments, or specific sections."\n            )\n        )\n)'

## 6. 🤖 Build & Run ReAct Agent

In [ ]:
tools = [vector_query_tool, summary_tool]
memory = ChatMemoryBuffer.from_defaults(token_limit=4096)

agent = ReActAgent(
    tools=tools,
    llm=Settings.llm,
    memory=memory,
    verbose=True,
)


response = await agent.run("prompt")
print(response)

Running step init_run
Step init_run produced event AgentInput
Running step setup_agent
Step setup_agent produced event AgentSetup
Running step run_agent_step


2026-03-05 00:14:48,093 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Step run_agent_step produced event AgentOutput
Running step parse_agent_output
Step parse_agent_output produced no event
Running step call_tool


2026-03-05 00:15:02,687 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Step call_tool produced event ToolCallResult
Running step aggregate_tool_results
Step aggregate_tool_results produced event AgentInput
Running step setup_agent
Step setup_agent produced event AgentSetup
Running step run_agent_step


2026-03-05 00:15:04,122 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Step run_agent_step produced event AgentOutput
Running step parse_agent_output
Step parse_agent_output produced event StopEvent
The paper titled "Improving Retrieval-Augmented Generation through Multi-Agent Reinforcement Learning" discusses how to enhance retrieval-augmented generation (RAG) systems by using a multi-agent reinforcement learning framework. This approach aims to optimize various components of the system collaboratively, ultimately improving the accuracy and quality of answers generated in question-answering tasks.


## 7. 📊 Evaluation (Next Step)

Evaluation is done by the help of the clause ai 

In [13]:
from llama_index.core.evaluation import BatchEvalRunner
import pandas as pd

# Your test questions (with optional ground truth)
eval_dataset = [
    {
        "query": "What is the main contribution of this paper?",
        "reference": "A multi-agent RL framework for optimizing RAG systems."
    },
    {
        "query": "What datasets were used for evaluation?",
        "reference": "The paper evaluates on TriviaQA and HotpotQA."
    },
    {
        "query": "What were the baseline models compared against?",
        "reference": None   # No reference — skip correctness eval
    },
    {
        "query": "Summarize the results section.",
        "reference": None
    },
]

async def batch_evaluate(eval_dataset: list):
    """Run all evaluators across all questions."""

    all_results = []

    for item in eval_dataset:
        query     = item["query"]
        reference = item.get("reference")

        print(f"Evaluating: {query[:50]}...")
        result = await evaluate_single_response(query, reference)
        result["query"] = query
        all_results.append(result)

    return all_results


# Run batch evaluation
all_results = await batch_evaluate(eval_dataset)


# ── Format as DataFrame ───────────────────────────────────
rows = []
for r in all_results:
    row = {"query": r["query"]}
    for metric in ["faithfulness", "answer_relevancy", "context_relevancy", "correctness"]:
        if metric in r:
            row[f"{metric}_score"]   = r[metric]["score"]
            row[f"{metric}_passing"] = r[metric]["passing"]
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string())

# ── Summary Stats ─────────────────────────────────────────
print("\n── Average Scores ──")
score_cols = [c for c in df.columns if c.endswith("_score")]
print(df[score_cols].mean())

Evaluating: What is the main contribution of this paper?...
Running step init_run
Step init_run produced event AgentInput
Running step setup_agent
Step setup_agent produced event AgentSetup
Running step run_agent_step


2026-03-05 00:34:28,503 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Step run_agent_step produced event AgentOutput
Running step parse_agent_output
Step parse_agent_output produced no event
Running step call_tool
Step call_tool produced event ToolCallResult
Running step aggregate_tool_results
Step aggregate_tool_results produced event AgentInput
Running step setup_agent
Step setup_agent produced event AgentSetup
Running step run_agent_step


2026-03-05 00:34:30,196 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Step run_agent_step produced event AgentOutput
Running step parse_agent_output
Step parse_agent_output produced no event
Running step call_tool


...

Step call_tool produced event ToolCallResult
Running step aggregate_tool_results
Step aggregate_tool_results produced event AgentInput
Running step setup_agent
Step setup_agent produced event AgentSetup
Running step run_agent_step


2026-03-05 00:35:24,215 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Step run_agent_step produced event AgentOutput
Running step parse_agent_output
Step parse_agent_output produced event StopEvent


2026-03-05 00:35:33,022 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


                                             query  faithfulness_score  faithfulness_passing  answer_relevancy_score answer_relevancy_passing context_relevancy_score context_relevancy_passing  correctness_score correctness_passing
0     What is the main contribution of this paper?                 0.0                 False                     1.0                     None                    None                      None                4.0                True
1          What datasets were used for evaluation?                 0.0                 False                     1.0                     None                    None                      None                3.0               False
2  What were the baseline models compared against?                 0.0                 False                     1.0                     None                    None                      None                NaN                 NaN
3                   Summarize the results section.                 0.0      